> ⏸ **Parked: not part of the current phase.** This notebook will be updated once model training and evaluation (notebooks 01–03) are finalised. Don't run it yet.

# 07 · Demo UI

Runs the Gradio demo (upload → verdict → Grad-CAM++ overlay → cited explanation). Kaggle can't serve localhost, so it prints a temporary public `gradio.live` link. Stop the cell to shut it down.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 (not P100)
- **Internet:** On
- **Inputs:** output of **02_train**
- **Secrets (Add-ons → Secrets):** one of `ANTHROPIC_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY` or `LLM_API_KEY` (none needed for Ollama) (optional)

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup, setup_llm, start_ollama
setup(data=False, checkpoint=True)

# --- LLM backend -------------------------------------------------------------------------
LLM_PROVIDER = "anthropic"   # anthropic | openrouter | ollama | openai (any OpenAI-compatible server)
LLM_MODEL = None             # None = provider default; e.g. "meta-llama/llama-3.3-70b-instruct" on OpenRouter
if LLM_PROVIDER == "ollama":
    start_ollama(LLM_MODEL or "llama3.1:8b")   # local model running inside this Kaggle session
setup_llm(LLM_PROVIDER, LLM_MODEL)
!pip install -q gradio

In [ ]:
!python app.py --share